# Notebook 03 - Feature Engineering (Customer-Level)

**Input:** `data/interim/transactions_customer_level.parquet` (~800K rows, 5,861 customers)

**Output:** `data/processed/customer_features.parquet` - one row per customer with ~25 features ready for segmentation, CLV, and churn modeling

## Why this notebook is the most important one

In ML, you'll often hear:*"better features beat better models."* Spending an extra day here saves a week of trying to squeeze accuracy out of model tuning

## The critical concept: snapshot date and temporal splits

**Wrong way:** Compute all features over the entire dateset, then try to predict "will customer churn". This **leaks the future into the features** - your features know what hasn't happened yet from the model's perspective.

**Right way:** Pick a `snapshot_date`. Compute features using ONLY data up to that date. Define the target using data AFTER that date. The model learns to predict the unknown future from the known past - exactly how it will operate when deployed.

Our data ends `2011-12-09`. We'll use:
- **Feature window:** all data up to `2011-09-09` (snapshot date)
- **Target window:** `2011-09-10` to `2011-12-09` (next 90 days)

Customers who first purchased AFTER the snapshot date can't be features-engineered (no history). We exclude them here.

## Feature groups we'll build

1. **RFM** — Recency, Frequency, Monetary (the foundational triad)
2. **Behavioral** — AOV, basket size, days between orders, consistency
3. **Temporal/trend** — recent vs. historical activity, momentum
4. **Product mix** — diversity, category breadth (proxied by stock-code count)
5. **Geography** — country, is_uk flag
6. **Return behavior** — return rate, return count (from cancellations table)
7. **Targets** — for downstream supervised tasks:
   - `target_revenue_90d` (regression target for CLV)
   - `target_purchased_90d` (binary, classification target for churn)

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_style('whitegrid')

INTERIM_DIR = Path('../data/interim')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load data and efine the tme split

We'll define snapshot date by working backward from the data's max date. Using **90 days** as our target window is a defensible choice <br>
for an e-commerce business - long enough to capture re-purchase behavior, short enough to be actionable.

In [2]:
df = pd.read_parquet(INTERIM_DIR / 'transactions_customer_level.parquet')
cancellations = pd.read_parquet(INTERIM_DIR / 'cancellations.parquet')

data_max_date = df['InvoiceDate'].max()
TARGET_WINDOW_DAYS = 90
snapshot_date = data_max_date - timedelta(days=TARGET_WINDOW_DAYS)

print(f'Data spans:     {df['InvoiceDate'].min()} -> {data_max_date}')
print(f'Snapshot date:  {snapshot_date}')
print(f'Feature window: start -> {snapshot_date}')
print(f'Target window:  {snapshot_date} -> {data_max_date} ({TARGET_WINDOW_DAYS} days)')


Data spans:     2009-12-01 07:45:00 -> 2011-12-09 12:50:00
Snapshot date:  2011-09-10 12:50:00
Feature window: start -> 2011-09-10 12:50:00
Target window:  2011-09-10 12:50:00 -> 2011-12-09 12:50:00 (90 days)


In [3]:
# Split Data
feat_df = df[df['InvoiceDate'] <= snapshot_date].copy()
target_df = df[df['InvoiceDate'] > snapshot_date].copy()

print(f'Feature window rows: {len(feat_df):,}')
print(f'Target window rows: {len(target_df):,}')
print()
print(f'Customers visible in feature window: {feat_df['CustomerID'].nunique():,}')
print(f'Customers visible in target window: {target_df['CustomerID'].nunique():,}')
print(f'Customers in both (will become positives for churn target): '
      f'{len(set(feat_df['CustomerID']) & set(target_df['CustomerID'])):,}')

Feature window rows: 641,705
Target window rows: 160,932

Customers visible in feature window: 5,256
Customers visible in target window: 2,885
Customers in both (will become positives for churn target): 2,289


## 2. Initialize the feature table

Start with one row per customer who exists in the feature window. We'll join features onto this skeleton as we build them.

In [4]:
features = pd.DataFrame({'CustomerID': feat_df['CustomerID'].unique()}).sort_values('CustomerID').reset_index(drop=True)
print(f'Customer skeleton: {len(features):,} rows')
features.head()

Customer skeleton: 5,256 rows


,CustomerID
0,12346.0
1,12347.0
2,12348.0
3,12349.0
4,12350.0


## 3. RFM features

**Recency** - Days since last purchase as of snapshot date. Lower is better (more recent = more likely to repeat)

**Frequency** - Number of distinct invoices in the feature window. Higher = more engaged.

**Monetary** - Total revenue in feature window. The classic measure of customer value.

In [5]:
rfm = feat_df.groupby('CustomerID').agg(
    last_purchase_date=('InvoiceDate', 'max'),
    first_purchase_date=('InvoiceDate', 'min'),
    frequency=('Invoice', 'nunique'),
    monetary=('Revenue', 'sum'),
    total_units=('Quantity', 'sum'),
    total_line_items=('Invoice', 'count')
).reset_index()

rfm['recency_days'] = (snapshot_date - rfm['last_purchase_date']).dt.days
rfm['tenure_days'] = (rfm['last_purchase_date'] - rfm['first_purchase_date']).dt.days
rfm['customer_age_days'] = (snapshot_date - rfm['first_purchase_date']).dt.days

features = features.merge(rfm, on='CustomerID', how='left')
features[['CustomerID', 'recency_days', 'frequency', 'monetary', 'tenure_days']].describe()

,recency_days,frequency,monetary,tenure_days
count,"5,256.00","5,256.00","5,256.00","5,256.00"
mean,206.36,5.71,"2,672.38",224.43
std,174.35,11.22,"12,407.43",218.60
min,0.00,1.00,1.55,0.00
25%,49.00,1.00,322.29,0.00
50%,163.00,3.00,798.00,171.00
75%,324.00,6.00,"2,106.93",418.00
max,648.00,284.00,"484,615.10",646.00


## 4. Behavioral features

**AOV (Average Oder Value)** -- total revenue ÷ frequency. Distinguishes "big spender per order" from "small but loyal".

**Average basket size** -- units per order. B2B buyers have larger baskets.

**Average days between orders** -- order cadence. Critical for understanding the customer's natural buying rhythm.

**Inter-order std** -- consistency of the buying rhythm. Predictable customers are easier to forecast.

**Note:** For one-order customers, `avg_days_between_orders` and its std are undefined. We'll fill NaN with the customer's tenure <br>
(proxy for "at least this long since they last needed to buy").

In [6]:
# Compute order-level aggregates first (one row per invoice)
orders = feat_df.groupby(['CustomerID','Invoice']).agg(
    order_value=('Revenue', 'sum'),
    order_units=('Quantity', 'sum'),
    order_distinct_products=('StockCode', 'nunique'),
    order_date=('InvoiceDate', 'min')
).reset_index()

# Now aggregate orders to customer level
behavioral = orders.groupby('CustomerID').agg(
    avg_order_value=('order_value', 'mean'),
    median_order_value=('order_value', 'median'),
    std_order_value=('order_value', 'std'),
    max_order_value=('order_value', 'max'),
    avg_basket_size=('order_units', 'mean'),
    avg_distinct_products_per_order=('order_distinct_products', 'mean')
).reset_index()

features = features.merge(behavioral, on='CustomerID', how='left')

# Days between consecutive orders
orders_sorted = orders.sort_values(['CustomerID','order_date'])
orders_sorted['days_since_prev_order'] = (
    orders_sorted.groupby('CustomerID')['order_date'].diff().dt.days
)

cadence = orders_sorted.groupby('CustomerID')['days_since_prev_order'].agg(
    avg_days_between_orders='mean',
    std_days_between_orders='std',
    max_days_between_orders='max',
).reset_index()

features = features.merge(cadence, on='CustomerID', how='left')

# Santity check: one-order customers should have NaN for cadence features
print('One-order customers (no cadence):', features['avg_days_between_orders'].isna().sum())
print('Multi-order customers (cadence valid):', features['avg_days_between_orders'].notna().sum())

features[['avg_order_value','avg_basket_size','avg_days_between_orders']].describe()

One-order customers (no cadence): 1575
Multi-order customers (cadence valid): 3681


,avg_order_value,avg_basket_size,avg_days_between_orders
count,"5,256.00","5,256.00","3,681.00"
mean,377.87,254.06,90.13
std,640.29,"1,448.31",81.26
min,1.55,1.00,0.00
25%,178.63,90.73,38.00
50%,282.37,153.24,67.12
75%,417.98,259.81,116.40
max,"25,784.32","87,167.00",632.00


## 5. Product-mix features

**Unique products bought** - proxy for product-line breadth

**Diversity score** - unique products ÷ total line items. Closer to 1 = explores variety; closer to 0 = repeats same products.

Why this matters: a customer who keeps re-buying the same product is more predictable but limited in growth potential.<br>
A customer who explores has higher growth potential but is harder to forecast.

In [8]:
product_mix = feat_df.groupby('CustomerID').agg(
    unique_products=('StockCode','nunique')
).reset_index()

features = features.merge(product_mix, on='CustomerID', how='left')
features['product_diversity'] = features['unique_products'] / features['total_line_items'].clip(lower=1)

features[['unique_products','product_diversity']].describe()

,unique_products,product_diversity
count,"5,256.00","5,256.00"
mean,74.55,0.82
std,104.98,0.19
min,1.00,0.04
25%,18.00,0.70
50%,41.00,0.87
75%,92.00,1.00
max,"2,178.00",1.00


## 6. Temporal trend features

These compare a customer's recent activity to their historical baseline. **Trend features are often the strongest predictors of churn** - a customer slowing recently is much more likely to lapse than one going at their usual pace.

- **Revenue in the last 30/60/90 days** (within feature window)

- **Trend ratio** = recent_30days_revenue / prior_30days_revenue. < 1 means slowing down, > 1 means accelerating.

- **Month activity** = distinct calendar months with any purchase

In [9]:
# Define rolling windows ending at snapshot_date
windows = {
    'last_30d': feat_df[feat_df['InvoiceDate'] > snapshot_date - timedelta(days=30)],
    'last_60d': feat_df[feat_df['InvoiceDate'] > snapshot_date - timedelta(days=60)],
    'last_90d': feat_df[feat_df['InvoiceDate'] > snapshot_date - timedelta(days=90)],
    'prior_30d': feat_df[(feat_df['InvoiceDate'] > snapshot_date - timedelta(days=60)) &
                         (feat_df['InvoiceDate'] <= snapshot_date - timedelta(days=30))]
}

for name, window in windows.items():
    agg = window.groupby('CustomerID').agg(
        **{f'revenue_{name}': ('Revenue', 'sum'),
           f'orders_{name}': ('Invoice','nunique')},
    ).reset_index()
    features = features.merge(agg, on='CustomerID', how='left')
    
# Fill NaN with 0 (no activity in that window = 0 revenue/orders)
trend_cols = [c for c in features.columns if c.startswith(('revenue_last', 'orders_last', 'revenue_prior', 'orders_prior'))]
features[trend_cols] = features[trend_cols].fillna(0)

# Trend ratio (last 30 vs. prior 30). Add small epsilon to avoid div-by-zero
features['revenue_trend_ratio'] = (
    features['revenue_last_30d'] / (features['revenue_prior_30d'] + 0.01)
)

# Months active
feat_df['YearMonth'] = feat_df['InvoiceDate'].dt.to_period('M')
months_active = feat_df.groupby('CustomerID')['YearMonth'].nunique().reset_index(name='months_active')
features = features.merge(months_active, on='CustomerID', how='left')

features[['revenue_last_30d','revenue_last_60d','revenue_last_90d','revenue_trend_ratio','months_active']].describe()

,revenue_last_30d,revenue_last_60d,revenue_last_90d,revenue_trend_ratio,months_active
count,"5,256.00","5,256.00","5,256.00","5,256.00","5,256.00"
mean,117.59,240.32,347.69,"4,501.30",4.00
std,771.19,"1,365.04","1,946.47","22,578.78",4.02
min,0.00,0.00,0.00,0.00,1.00
25%,0.00,0.00,0.00,0.00,1.00
50%,0.00,0.00,0.00,0.00,2.00
75%,0.00,136.38,266.09,0.00,5.00
max,"29,775.78","44,314.22","62,151.70","848,900.00",22.00


## 7. Geographic features

From EDA we know UK is 92% of revenue. We use the customer's **primary country** (most common in their history) and a simple `is_uk` flag.<br>
International customers are a small but distinct segment.

In [19]:
primary_country = (
    feat_df.groupby('CustomerID')['Country']
        .agg(lambda x: x.mode().iloc[0])
        .reset_index(name='primary_country')
)

# Ensure a clean merge if this cell is re-run in the notebook
features = features.drop(columns=['primary_country', 'primary_country_x', 'primary_country_y'], errors='ignore')
features = features.merge(primary_country, on='CustomerID', how='left', validate='one_to_one')
features['is_uk'] = (features['primary_country'] == 'United Kingdom').astype(int)

print('Country distribution:')
features['primary_country'].value_counts().head(10)


Country distribution:


primary_country
United Kingdom    4799
Germany             92
France              76
Spain               30
Belgium             27
Netherlands         22
Switzerland         20
Portugal            19
Sweden              19
Italy               14
Name: count, dtype: int64

## 8. Return behavior features

From the `cancellations.parquet` table. We compute return rate within the feature window only (no future leakage)

In [22]:
# Cancellation within feature window with valid CustomerID
cancel_feat = cancellations[
    (cancellations['InvoiceDate'] <= snapshot_date) &
    cancellations['CustomerID'].notna()
].copy()
cancel_feat['return_value'] = cancel_feat['Quantity'].abs() * cancel_feat['Price'].abs()

returns = cancel_feat.groupby('CustomerID').agg(
    return_count=('Invoice', 'nunique'),
    return_value=('return_value', 'sum')
).reset_index()

features = features.merge(returns, on='CustomerID', how='left')
features['return_count'] = features['return_count'].fillna(0).astype(int)
features['return_value'] = features['return_value'].fillna(0)

# Return rate = return_value / total monetary value
features['return_rate'] = features['return_value'] / (features['monetary']+ 0.01)

print(f'Customers with any returns: {(features["return_count"] > 0).sum():,}')
features[['return_count','return_value','return_rate']].describe()

Customers with any returns: 2,223


,return_count,return_value,return_rate
count,"5,256.00","5,256.00","5,256.00"
mean,1.26,129.82,0.03
std,3.38,"1,500.84",0.29
min,0.00,0.00,0.00
25%,0.00,0.00,0.00
50%,0.00,0.00,0.00
75%,1.00,29.75,0.02
max,91.00,"77,621.14",18.62


## 9. Build the targets

Two targets, both computed from the target window (data AFTER snapshot date):
1. `target_revenue_90d` - total revenue in next 90 days. Regression target for CLV models.
2. `target_purchased_90d` - binary, did they purchase at all. Classification target for churn (where target=0 means churned).

In [24]:
targets = target_df.groupby('CustomerID').agg(
    target_revenue_90d=('Revenue', 'sum'),
    target_orders_90d=('Invoice','nunique')
).reset_index()

features = features.merge(targets, on='CustomerID', how='left')
features['target_revenue_90d'] = features['target_revenue_90d'].fillna(0)
features['target_orders_90d'] = features['target_orders_90d'].fillna(0).astype(int)
features['target_purchased_90d'] = (features['target_orders_90d'] > 0).astype(int)

print('Target Distribution:')
print(f'    Purchased in next 90 days: {features["target_purchased_90d"].sum():,} '
      f'({features["target_purchased_90d"].mean()*100:.1f}%)')
print(f'    Churned (no purchase 90 days): {(features["target_purchased_90d"] == 0).sum():,} '
      f'({(features["target_purchased_90d"] == 0).mean()*100:.1f}%)')
print()
print('Revenue taarget stats (next 90 days):')
features['target_revenue_90d'].describe()


Target Distribution:
    Purchased in next 90 days: 2,289 (43.6%)
    Churned (no purchase 90 days): 2,967 (56.4%)

Revenue taarget stats (next 90 days):


count     5,256.00
mean        570.81
std       3,984.40
min           0.00
25%           0.00
50%           0.00
75%         434.99
max     168,469.60
Name: target_revenue_90d, dtype: float64